In [ ]:
%load_ext autoreload
%autoreload 2
import os
import multiprocessing as mp
import itertools

# import simulations
import pickle
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
from folktables import ACSDataSource


In [ ]:
OUTPUT_DIR = "../data"

features = [
    "SEX",
    "DIS",
    "ESP",
    "MIL",
    "NATIVITY",
    "DEAR",
    "DEYE",
    "DREM",
    "RAC1P",
    "PINCP",
]


def get_data(year, features, state='CA', root_dir='data'):
    data_source = ACSDataSource(survey_year=year, horizon='1-Year', survey='person', root_dir=root_dir)
    acs_data = data_source.get_data(states=[state], download=True)

    df = acs_data[acs_data['PINCP'].notna() & acs_data['PRIVCOV'].notna()]

    Y = (df['PRIVCOV'] == 1).astype(int).to_numpy()
    X_features = df[features].fillna(0)

    return X_features, Y


os.makedirs(f"{OUTPUT_DIR}/health_care", exist_ok=True)

for year in ["2017", "2018", "2019"]:
    print(f"Fetching data for {year}...")
    X_feat, Y_2018 = get_data(year, features)

    # Combine into one DF for saving
    df_save = X_feat.copy()
    df_save["generated_label_Y"] = Y_2018

    save_path = f"{OUTPUT_DIR}/health_care/features_{year}.parquet"
    print(f"Saving {save_path}...")
    df_save.to_parquet(save_path)